# SumCheck Optimization Journey — From Baseline to Fully Fused

> **ECE-9413 Assignment 2 — GPU Optimization Notebook**
>
> Each optimization level builds directly on the previous one.
> Every cell is runnable and benchmarked. Correctness is verified against the baseline at each step.

## Optimization Roadmap

| Level | Optimization | Key Technique | Expected Gain |
|---|---|---|---|
| 0 | **Baseline** | `student.py` as-is | Reference |
| 1 | **JIT Primitives** | `@jax.jit` on `mod_mul/add/sub` | Eliminate Python dispatch |
| 2 | **Fused MLE Update** | Single JIT expression for `(o-z)*t+z` | 3× fewer HBM reads |
| 3 | **Stacked Tables** | `(V, N)` array instead of dict | Clean JIT tracing |
| 4 | **Reshape Split** | `reshape(-1,2)` instead of `[0::2]` | Coalesced GPU access |
| 5 | **vmap Degree Loop** | Vectorize `g(0)..g(d)` simultaneously | Batch kernel launches |
| 6 | **JIT Full Round** | Single JIT over split+MLE+eval+reduce | Maximum kernel fusion |
| 7 | **JIT Full SumCheck** | JIT the entire function | Eliminate all Python overhead |
| 8 | **lax.scan Outer Loop** | XLA-native round loop (fixed-buffer) | True loop without unrolling |
| 9 | **Combined Production** | All optimizations stacked | Maximum throughput |

---

### Why SumCheck is memory-bandwidth bound

```
Arithmetic Intensity of mod_mul_32:
  FLOPs per element : 2  (one multiply + one mod)
  Bytes per element : 12 (read a, read b, write result — 4 bytes each)
  AI = 2/12 ≈ 0.17 FLOPs/byte

A100 ridge point ≈ 200 FLOPs/byte
→ SumCheck is 1000× below the ridge — ALWAYS memory-bound.
→ Every optimization must reduce bytes moved, not FLOPs.
```

---
## Section 0 — Setup

In [ ]:
import jax
import jax.numpy as jnp
from jax import lax
import time
import functools

jax.config.update("jax_enable_x64", True)

print("JAX version :", jax.__version__)
print("Backend     :", jax.default_backend())
print("Devices     :", jax.devices())

Q32    = jnp.uint32(4_294_967_291)   # largest 32-bit prime
Q_tiny = jnp.uint32(17)              # readable examples

# ── Benchmark harness ─────────────────────────────────────────────────────────
def bench(fn, *args, warmup=3, runs=10, label="", **kwargs):
    """Time fn(*args, **kwargs). Returns mean ms and last result."""
    for _ in range(warmup):
        result = fn(*args, **kwargs)
        # block on the JAX array so compile happens during warmup
        jax.tree_util.tree_map(lambda x: x.block_until_ready() if hasattr(x, 'block_until_ready') else x, result)
    t0 = time.perf_counter()
    for _ in range(runs):
        result = fn(*args, **kwargs)
        jax.tree_util.tree_map(lambda x: x.block_until_ready() if hasattr(x, 'block_until_ready') else x, result)
    ms = (time.perf_counter() - t0) / runs * 1000
    print(f"  {label:50s}  {ms:8.2f} ms")
    return ms, result

# ── Correctness checker ────────────────────────────────────────────────────────
_REFERENCE = None   # filled in after baseline section

def check(claim0, round_evals, label=""):
    global _REFERENCE
    if _REFERENCE is None:
        _REFERENCE = (int(claim0), round_evals.tolist())
        print(f"  [{label}] Reference set: claim0={int(claim0)}, evals={round_evals.tolist()}")
        return True
    ref_c0, ref_re = _REFERENCE
    ok = (int(claim0) == ref_c0) and (round_evals.tolist() == ref_re)
    print(f"  [{label}] {'PASS' if ok else 'FAIL'}")
    return ok

# ── Tiny reference test (Q=17, all_ones) ──────────────────────────────────────
TINY_TABLES  = {'a': jnp.array([0,1,2,3,4,5,6,7], dtype=jnp.uint32)}
TINY_CHALS   = jnp.array([8, 5], dtype=jnp.uint32)   # r1=8, r2=5
TINY_EXPECTED_C0 = 11
TINY_EXPECTED_RE = [[12, 16], [3, 7], [1, 5]]

# ── Large benchmark tables (n=20 rounds, expression a*b*c, Q32) ───────────────
N_BENCH = 20
N       = 2**N_BENCH
key     = jax.random.PRNGKey(42)
BENCH_EXPR  = [['a', 'b', 'c']]
BENCH_VARS  = ['a', 'b', 'c']
BENCH_TABLES = {v: jax.random.randint(key, (N,), 0, int(Q32), dtype=jnp.uint32)
                for v in BENCH_VARS}
BENCH_CHALS  = jax.random.randint(key, (N_BENCH-1,), 1, int(Q32), dtype=jnp.uint32)

print(f"\nBench config: n={N_BENCH}, N=2^{N_BENCH}={N:,}, expr=a*b*c")
print(f"Per-var table: {N*4/1024**2:.1f} MB   Total: {N*4*3/1024**2:.1f} MB")

---
## Level 0 — Baseline (`student.py`)

Exact copy of the submission baseline. Three separate kernels per MLE update,
Python dict for tables, Python loops over rounds and over `t = 0..degree`.

**Hotspots to fix (in order of impact):**
1. `mle_update_32` calls 3 unjitted functions → 3 kernel launches, 9× HBM traffic
2. Python dict prevents clean JIT tracing inside rounds
3. `table[0::2]` stride-2 gather → non-coalesced GPU reads
4. Python loop `for t in range(degree+1)` → sequential kernel launches
5. Python loop `for round_idx in range(num_rounds)` → n rounds of Python overhead

In [ ]:
# ── Level 0: Exact baseline from student.py ───────────────────────────────────

def mod_add_32_v0(a, b, q):
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return ((a64 + b64) % q64).astype(jnp.uint32)

def mod_sub_32_v0(a, b, q):
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return ((a64 + q64 - b64) % q64).astype(jnp.uint32)

def mod_mul_32_v0(a, b, q):
    a64 = jnp.asarray(a, dtype=jnp.uint64)
    b64 = jnp.asarray(b, dtype=jnp.uint64)
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return ((a64 * b64) % q64).astype(jnp.uint32)

def mle_update_32_v0(zero_eval, one_eval, target_eval, *, q):
    # 3 SEPARATE unjitted calls → 3 kernel launches, 9× HBM traffic
    diff   = mod_sub_32_v0(one_eval, zero_eval, q)
    scaled = mod_mul_32_v0(diff, target_eval, q)
    return mod_add_32_v0(scaled, zero_eval, q)

def _eval_composition_v0(var_at_t, expression, *, q):
    some_arr = next(iter(var_at_t.values()))
    total = jnp.zeros(some_arr.shape, dtype=jnp.uint32)
    for term in expression:
        prod = jnp.ones(some_arr.shape, dtype=jnp.uint32)
        for var_name in term:
            prod = mod_mul_32_v0(prod, var_at_t[var_name], q)
        total = mod_add_32_v0(total, prod, q)
    return total

def sumcheck_v0(eval_tables, *, q, expression, challenges, num_rounds):
    """Level 0: exact baseline from student.py."""
    tables = {name: jnp.asarray(val, dtype=jnp.uint32)
              for name, val in eval_tables.items()}
    degree = max(len(term) for term in expression)
    q64    = jnp.asarray(q, dtype=jnp.uint64)
    all_round_evals = []

    for round_idx in range(num_rounds):
        evens = {name: tables[name][0::2] for name in tables}  # stride-2 gather
        odds  = {name: tables[name][1::2] for name in tables}

        g_t_vals = []
        for t in range(degree + 1):
            if t == 0:
                var_at_t = evens
            elif t == 1:
                var_at_t = odds
            else:
                t_jax = jnp.asarray(t, dtype=jnp.uint32)
                var_at_t = {name: mle_update_32_v0(evens[name], odds[name], t_jax, q=q)
                            for name in tables}
            per_pair = _eval_composition_v0(var_at_t, expression, q=q)
            g_t = (jnp.sum(per_pair.astype(jnp.uint64)) % q64).astype(jnp.uint32)
            g_t_vals.append(g_t)
        all_round_evals.append(g_t_vals)

        if round_idx < num_rounds - 1:
            r = challenges[round_idx]
            tables = {name: mle_update_32_v0(evens[name], odds[name], r, q=q)
                      for name in tables}

    claim0 = mod_add_32_v0(all_round_evals[0][0], all_round_evals[0][1], q)
    round_evals = jnp.stack([jnp.stack(g_t) for g_t in all_round_evals])
    return claim0, round_evals


# Correctness on tiny example
c0, re = sumcheck_v0(TINY_TABLES, q=Q_tiny, expression=[['a']],
                     challenges=TINY_CHALS, num_rounds=3)
print("Tiny test:", "PASS" if int(c0)==TINY_EXPECTED_C0 and re.tolist()==TINY_EXPECTED_RE else "FAIL")

# Benchmark
print("\nBenchmark (n=20, a*b*c):")
ms_v0, (c0_ref, re_ref) = bench(
    sumcheck_v0, BENCH_TABLES, q=Q32, expression=BENCH_EXPR,
    challenges=BENCH_CHALS, num_rounds=N_BENCH,
    label="v0 baseline"
)
_REFERENCE = (int(c0_ref), re_ref.tolist())
print(f"  claim0={int(c0_ref)}")

---
## Level 1 — JIT the Primitives

**What changes:** Wrap `mod_add_32`, `mod_sub_32`, `mod_mul_32` with `@jax.jit`.

**Why it helps:**
- Without JIT, every call dispatches through Python → C++ → GPU driver → GPU. Each dispatch costs ~10–50 µs.
- With JIT, JAX compiles the function to an XLA HLO program on first call. Subsequent calls skip Python entirely.
- XLA can also fuse consecutive elementwise ops: `a+b` followed immediately by `%q` becomes one GPU instruction.

**Memory traffic:** Same number of bytes moved — this is a latency win, not a bandwidth win.

```
Without JIT:  Python overhead per call ≈ 20-50 µs
              n=20 rounds × (degree+1) × V ops = ~1000 dispatches → ~20-50 ms Python overhead
With JIT:     Python overhead per call ≈ 2-5 µs
              Same dispatch count → ~2-5 ms Python overhead
```

In [ ]:
# ── Level 1: JIT-compile the primitives ───────────────────────────────────────
# Single change from v0: add @jax.jit to each primitive.
# Everything else is identical.

@jax.jit
def mod_add_32_v1(a, b, q):
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return ((a.astype(jnp.uint64) + b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

@jax.jit
def mod_sub_32_v1(a, b, q):
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return ((a.astype(jnp.uint64) + q64 - b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

@jax.jit
def mod_mul_32_v1(a, b, q):
    q64 = jnp.asarray(q, dtype=jnp.uint64)
    return ((a.astype(jnp.uint64) * b.astype(jnp.uint64)) % q64).astype(jnp.uint32)

def mle_update_32_v1(zero_eval, one_eval, target_eval, *, q):
    # Still 3 kernel launches, but each is JIT-compiled
    diff   = mod_sub_32_v1(one_eval, zero_eval, q)
    scaled = mod_mul_32_v1(diff, target_eval, q)
    return mod_add_32_v1(scaled, zero_eval, q)

def _eval_composition_v1(var_at_t, expression, *, q):
    some_arr = next(iter(var_at_t.values()))
    total = jnp.zeros(some_arr.shape, dtype=jnp.uint32)
    for term in expression:
        prod = jnp.ones(some_arr.shape, dtype=jnp.uint32)
        for var_name in term:
            prod = mod_mul_32_v1(prod, var_at_t[var_name], q)
        total = mod_add_32_v1(total, prod, q)
    return total

def sumcheck_v1(eval_tables, *, q, expression, challenges, num_rounds):
    """Level 1: JIT-compiled primitives, rest identical to v0."""
    tables = {name: jnp.asarray(val, dtype=jnp.uint32)
              for name, val in eval_tables.items()}
    degree = max(len(term) for term in expression)
    q64    = jnp.asarray(q, dtype=jnp.uint64)
    all_round_evals = []

    for round_idx in range(num_rounds):
        evens = {name: tables[name][0::2] for name in tables}
        odds  = {name: tables[name][1::2] for name in tables}

        g_t_vals = []
        for t in range(degree + 1):
            if t == 0:
                var_at_t = evens
            elif t == 1:
                var_at_t = odds
            else:
                t_jax = jnp.asarray(t, dtype=jnp.uint32)
                var_at_t = {name: mle_update_32_v1(evens[name], odds[name], t_jax, q=q)
                            for name in tables}
            per_pair = _eval_composition_v1(var_at_t, expression, q=q)
            g_t = (jnp.sum(per_pair.astype(jnp.uint64)) % q64).astype(jnp.uint32)
            g_t_vals.append(g_t)
        all_round_evals.append(g_t_vals)

        if round_idx < num_rounds - 1:
            r = challenges[round_idx]
            tables = {name: mle_update_32_v1(evens[name], odds[name], r, q=q)
                      for name in tables}

    claim0 = mod_add_32_v1(all_round_evals[0][0], all_round_evals[0][1], q)
    round_evals = jnp.stack([jnp.stack(g_t) for g_t in all_round_evals])
    return claim0, round_evals


# Verify + benchmark
c0, re = sumcheck_v1(TINY_TABLES, q=Q_tiny, expression=[['a']],
                     challenges=TINY_CHALS, num_rounds=3)
check(c0, re, "v1")

print("\nBenchmark:")
ms_v1, _ = bench(
    sumcheck_v1, BENCH_TABLES, q=Q32, expression=BENCH_EXPR,
    challenges=BENCH_CHALS, num_rounds=N_BENCH,
    label="v1 jit primitives"
)
print(f"  Speedup vs v0: {ms_v0/ms_v1:.2f}×")

---
## Level 2 — Fused MLE Update

**What changes:** The three operations inside `mle_update_32` are merged into one `@jax.jit` expression.

**Why it matters:**  
MLE update is called `O(n)` times per sumcheck run — it dominates the memory traffic.

```
Unfused (v1):  diff = sub(o, z)       → write diff  to HBM  (N/2 × 4B)
               scaled = mul(diff, t)  → read diff, write scaled  (2 × N/2 × 4B)
               result = add(scaled,z) → read scaled, read z, write result
               Total: 9 × (N/2) × 4 bytes moved

Fused (v2):    XLA sees (o-z)*t+z as a single expression graph.
               diff and scaled stay in GPU registers / shared memory.
               Total: 3 × (N/2) × 4 bytes moved  (read o, read z, write result)
               Savings: 3× less HBM traffic per MLE call
```

This is the single biggest win because:
1. MLE update is the innermost hot loop
2. It's called `(n + degree - 1) × n / 2` times total across all rounds
3. The 3× bandwidth saving directly translates to 3× more throughput

In [ ]:
# ── Level 2: Fused MLE update ─────────────────────────────────────────────────
# Single change from v1: mle_update_32 becomes one @jax.jit expression.
# The primitives mod_add/sub/mul stay the same (still used elsewhere).

# Keep v1 primitives
mod_add_32_v2 = mod_add_32_v1
mod_sub_32_v2 = mod_sub_32_v1
mod_mul_32_v2 = mod_mul_32_v1

@jax.jit
def mle_update_32_v2(zero_eval, one_eval, target_eval, q):
    """
    Fused: sub + mul + add as ONE JIT expression.
    XLA keeps diff and scaled in registers — never written to HBM.
    Result: 1 kernel instead of 3, 3× less HBM traffic.
    
    Note: q is a positional arg (not keyword) so jax.jit can trace it as
    a static scalar without needing static_argnames.
    """
    q64     = jnp.uint64(q)
    z64     = zero_eval.astype(jnp.uint64)
    o64     = one_eval.astype(jnp.uint64)
    t64     = jnp.asarray(target_eval, dtype=jnp.uint64)
    diff    = (o64 + q64 - z64) % q64            # (o - z) mod q
    scaled  = (diff * t64) % q64                  # diff * t mod q
    result  = (scaled + z64) % q64                # + z mod q
    return result.astype(jnp.uint32)


# ── Demonstrate the bandwidth difference ──────────────────────────────────────
N_demo = 2**20
a_demo = jax.random.randint(key, (N_demo,), 0, int(Q32), dtype=jnp.uint32)
b_demo = jax.random.randint(key, (N_demo,), 0, int(Q32), dtype=jnp.uint32)
r_demo = jnp.uint32(12345678)

# Warm up both
_ = mle_update_32_v1(a_demo, b_demo, r_demo, q=Q32).block_until_ready()
_ = mle_update_32_v2(a_demo, b_demo, r_demo, Q32).block_until_ready()

RUNS_DEMO = 100
t0 = time.perf_counter()
for _ in range(RUNS_DEMO):
    mle_update_32_v1(a_demo, b_demo, r_demo, q=Q32).block_until_ready()
ms_unfused = (time.perf_counter()-t0)/RUNS_DEMO*1000

t0 = time.perf_counter()
for _ in range(RUNS_DEMO):
    mle_update_32_v2(a_demo, b_demo, r_demo, Q32).block_until_ready()
ms_fused = (time.perf_counter()-t0)/RUNS_DEMO*1000

bytes_unfused = 9 * N_demo * 4
bytes_fused   = 3 * N_demo * 4
bw_unfused = bytes_unfused / (ms_unfused/1000) / 1e9
bw_fused   = bytes_fused   / (ms_fused  /1000) / 1e9

print("MLE Update: Unfused vs Fused (N=2^20)")
print(f"  Unfused (3 kernels): {ms_unfused:.3f} ms  HBM={bytes_unfused/1e6:.0f} MB  eff_BW={bw_unfused:.1f} GB/s")
print(f"  Fused   (1 kernel):  {ms_fused:.3f} ms  HBM={bytes_fused/1e6:.0f} MB  eff_BW={bw_fused:.1f} GB/s")
print(f"  Speedup: {ms_unfused/ms_fused:.2f}×   HBM traffic: 3× reduction (9→3 passes)")


def _eval_composition_v2(var_at_t, expression, *, q):
    some_arr = next(iter(var_at_t.values()))
    total = jnp.zeros(some_arr.shape, dtype=jnp.uint32)
    for term in expression:
        prod = jnp.ones(some_arr.shape, dtype=jnp.uint32)
        for var_name in term:
            prod = mod_mul_32_v2(prod, var_at_t[var_name], q)
        total = mod_add_32_v2(total, prod, q)
    return total

def sumcheck_v2(eval_tables, *, q, expression, challenges, num_rounds):
    """Level 2: fused MLE update (1 kernel instead of 3)."""
    tables = {name: jnp.asarray(val, dtype=jnp.uint32)
              for name, val in eval_tables.items()}
    degree = max(len(term) for term in expression)
    q64    = jnp.asarray(q, dtype=jnp.uint64)
    all_round_evals = []

    for round_idx in range(num_rounds):
        evens = {name: tables[name][0::2] for name in tables}
        odds  = {name: tables[name][1::2] for name in tables}

        g_t_vals = []
        for t in range(degree + 1):
            if t == 0:
                var_at_t = evens
            elif t == 1:
                var_at_t = odds
            else:
                t_jax = jnp.asarray(t, dtype=jnp.uint32)
                var_at_t = {name: mle_update_32_v2(evens[name], odds[name], t_jax, q)
                            for name in tables}
            per_pair = _eval_composition_v2(var_at_t, expression, q=q)
            g_t = (jnp.sum(per_pair.astype(jnp.uint64)) % q64).astype(jnp.uint32)
            g_t_vals.append(g_t)
        all_round_evals.append(g_t_vals)

        if round_idx < num_rounds - 1:
            r = challenges[round_idx]
            tables = {name: mle_update_32_v2(evens[name], odds[name], r, q)
                      for name in tables}

    claim0 = mod_add_32_v2(all_round_evals[0][0], all_round_evals[0][1], q)
    round_evals = jnp.stack([jnp.stack(g_t) for g_t in all_round_evals])
    return claim0, round_evals


c0, re = sumcheck_v2(TINY_TABLES, q=Q_tiny, expression=[['a']],
                     challenges=TINY_CHALS, num_rounds=3)
check(c0, re, "v2")

print("\nBenchmark:")
ms_v2, _ = bench(
    sumcheck_v2, BENCH_TABLES, q=Q32, expression=BENCH_EXPR,
    challenges=BENCH_CHALS, num_rounds=N_BENCH,
    label="v2 fused MLE"
)
print(f"  Speedup vs v0: {ms_v0/ms_v2:.2f}×  vs v1: {ms_v1/ms_v2:.2f}×")

---
## Level 3 — Stacked Tables: Replace Dict with `(V, N)` Array

**What changes:** `eval_tables` dict of `V` arrays becomes a single 2-D `(V, N)` array.
Variable names are replaced by row indices.

**Why it matters:**

1. **JIT tracing**: A Python dict with string keys forces JAX to trace each variable's ops
   separately. With a stacked array, JAX sees one contiguous tensor — it can plan memory
   layout and scheduling across all V variables simultaneously.

2. **Memory layout**: All V variable slices for the same index are contiguous in memory.
   When computing `a[i]*b[i]*c[i]`, rows 0,1,2 are accessed together — better L2 cache reuse.

3. **Enables future vmaps**: Once tables are a 2-D array, `jax.vmap` over the variable
   dimension becomes straightforward (used in Level 5+).

```python
# Before (dict):
evens = {var: tables[var][0::2] for var in tables}   # V separate gathers
val   = tables['a'] * tables['b']                    # string lookup

# After (stacked (V,N)):
evens = stacked[:, 0::2]                             # single slice
val   = stacked[0] * stacked[1]                      # integer index
```

In [ ]:
# ── Level 3: Stacked (V, N) tables ────────────────────────────────────────────

def stack_tables(eval_tables, var_order):
    """Convert dict of arrays to (V, N) stacked array. Returns (stacked, var_order)."""
    return jnp.stack([eval_tables[v] for v in var_order], axis=0)


@functools.partial(jax.jit, static_argnames=['term_indices'])
def eval_expression_stacked(stacked_VN, q, term_indices):
    """
    Evaluate composed expression pointwise on stacked (V, N) tables.

    term_indices: tuple of tuples of ints  — which row of stacked_VN each factor uses
                  e.g. for a*b+c with var_order=['a','b','c']: ((0,1),(2,))
    Returns: (N,) uint32 array
    """
    q64 = jnp.uint64(q)
    total = jnp.zeros(stacked_VN.shape[1], dtype=jnp.uint64)
    for term in term_indices:
        tv = stacked_VN[term[0]].astype(jnp.uint64)
        for idx in term[1:]:
            tv = (tv * stacked_VN[idx].astype(jnp.uint64)) % q64
        total = (total + tv) % q64
    return total.astype(jnp.uint32)


def build_term_indices(expression, var_order):
    """Convert expression list[list[str]] to term_indices tuple[tuple[int]]."""
    vi = {v: i for i, v in enumerate(var_order)}
    return tuple(tuple(vi[v] for v in term) for term in expression)


def sumcheck_v3(eval_tables, *, q, expression, challenges, num_rounds):
    """Level 3: stacked (V,N) tables, integer term indices, JIT eval."""
    var_order    = sorted(eval_tables.keys())
    term_indices = build_term_indices(expression, var_order)
    stacked      = stack_tables(eval_tables, var_order)  # (V, N) uint32
    degree       = max(len(term) for term in expression)
    q64          = jnp.uint64(q)
    all_round_evals = []

    for round_idx in range(num_rounds):
        # Split: still stride-2 gather (fixed in Level 4)
        evens = stacked[:, 0::2]   # (V, N/2)
        odds  = stacked[:, 1::2]   # (V, N/2)

        g_t_vals = []
        for t in range(degree + 1):
            if t == 0:
                tables_t = evens
            elif t == 1:
                tables_t = odds
            else:
                t_scalar = jnp.uint32(t)
                # Apply fused MLE update to all V rows at once
                tables_t = mle_update_32_v2(evens, odds, t_scalar, q)
            per_pair = eval_expression_stacked(tables_t, q, term_indices)
            g_t = (jnp.sum(per_pair.astype(jnp.uint64)) % q64).astype(jnp.uint32)
            g_t_vals.append(g_t)
        all_round_evals.append(g_t_vals)

        if round_idx < num_rounds - 1:
            r       = challenges[round_idx]
            stacked = mle_update_32_v2(evens, odds, r, q)  # (V, N/2)

    claim0 = mod_add_32_v2(all_round_evals[0][0], all_round_evals[0][1], q)
    round_evals = jnp.stack([jnp.stack(g_t) for g_t in all_round_evals])
    return claim0, round_evals


# Correctness
c0, re = sumcheck_v3(TINY_TABLES, q=Q_tiny, expression=[['a']],
                     challenges=TINY_CHALS, num_rounds=3)
check(c0, re, "v3")

print("\nBenchmark:")
ms_v3, _ = bench(
    sumcheck_v3, BENCH_TABLES, q=Q32, expression=BENCH_EXPR,
    challenges=BENCH_CHALS, num_rounds=N_BENCH,
    label="v3 stacked tables"
)
print(f"  Speedup vs v0: {ms_v0/ms_v3:.2f}×  vs v2: {ms_v2/ms_v3:.2f}×")

---
## Level 4 — Reshape Split: Replace Stride-2 Gather

**What changes:** `stacked[:, 0::2]` and `stacked[:, 1::2]` replaced by `reshape` + contiguous slice.

**Why stride-2 gather is bad on GPU:**

GPU memory controller fetches 128-byte cache lines. When reading every other element (`[0::2]`),
it must fetch a cache line, use half the bytes, discard the rest, then fetch the next cache line.
**Effective bandwidth utilization: 50%.**

```
Stride-2 gather:           Reshape + contiguous slice:
Cache line 0: [e0, o0]     view = stacked.reshape(V, N//2, 2)
  → use e0, discard o0     evens = view[:, :, 0]   ← contiguous in last dim!
Cache line 1: [e1, o1]     odds  = view[:, :, 1]   ← also contiguous!
  → use e1, discard o1     100% cache line utilization
...
Effective BW: 50%           Effective BW: 100%
```

> **Key insight:** `reshape` in JAX/NumPy is a **zero-copy view** — it costs nothing.
> You pay the same bytes read, but the hardware can do it 2× faster.

In [ ]:
# ── Level 4: Reshape split instead of stride-2 gather ─────────────────────────

@jax.jit
def split_stride2(stacked_VN):
    """Old: stride-2 gather — non-coalesced."""
    return stacked_VN[:, 0::2], stacked_VN[:, 1::2]

@jax.jit
def split_reshape(stacked_VN):
    """New: reshape → contiguous slice — fully coalesced."""
    V, N = stacked_VN.shape
    view = stacked_VN.reshape(V, N // 2, 2)   # zero-copy view
    return view[:, :, 0], view[:, :, 1]        # contiguous in last axis


# Benchmark the split operation alone
N_test     = 2**22
stk_test   = jax.random.randint(key, (3, N_test), 0, int(Q32), dtype=jnp.uint32)
_ = split_stride2(stk_test)[0].block_until_ready()
_ = split_reshape(stk_test)[0].block_until_ready()

RUNS_SPLIT = 200
t0 = time.perf_counter()
for _ in range(RUNS_SPLIT):
    split_stride2(stk_test)[0].block_until_ready()
ms_s2 = (time.perf_counter()-t0)/RUNS_SPLIT*1000

t0 = time.perf_counter()
for _ in range(RUNS_SPLIT):
    split_reshape(stk_test)[0].block_until_ready()
ms_rs = (time.perf_counter()-t0)/RUNS_SPLIT*1000

bytes_r = 3 * N_test * 4  # read input, write 2 outputs
print(f"Split comparison (V=3, N=2^22):")
print(f"  Stride-2 gather:  {ms_s2:.3f} ms  eff_BW={bytes_r/(ms_s2/1000)/1e9:.1f} GB/s")
print(f"  Reshape + slice:  {ms_rs:.3f} ms  eff_BW={bytes_r/(ms_rs/1000)/1e9:.1f} GB/s")
print(f"  Speedup: {ms_s2/ms_rs:.2f}×")

# Verify same result
e1, o1 = split_stride2(stk_test)
e2, o2 = split_reshape(stk_test)
assert jnp.array_equal(e1, e2) and jnp.array_equal(o1, o2), "Split mismatch!"
print("  Correctness: PASS")


def sumcheck_v4(eval_tables, *, q, expression, challenges, num_rounds):
    """Level 4: reshape split — coalesced GPU memory access."""
    var_order    = sorted(eval_tables.keys())
    term_indices = build_term_indices(expression, var_order)
    stacked      = stack_tables(eval_tables, var_order)   # (V, N)
    degree       = max(len(term) for term in expression)
    q64          = jnp.uint64(q)
    V            = stacked.shape[0]
    all_round_evals = []

    for round_idx in range(num_rounds):
        # Reshape split: zero-copy view + contiguous slice
        N_k  = stacked.shape[1]
        view = stacked.reshape(V, N_k // 2, 2)
        evens = view[:, :, 0]    # (V, N_k//2) — coalesced
        odds  = view[:, :, 1]    # (V, N_k//2) — coalesced

        g_t_vals = []
        for t in range(degree + 1):
            if t == 0:
                tables_t = evens
            elif t == 1:
                tables_t = odds
            else:
                tables_t = mle_update_32_v2(evens, odds, jnp.uint32(t), q)
            per_pair = eval_expression_stacked(tables_t, q, term_indices)
            g_t = (jnp.sum(per_pair.astype(jnp.uint64)) % q64).astype(jnp.uint32)
            g_t_vals.append(g_t)
        all_round_evals.append(g_t_vals)

        if round_idx < num_rounds - 1:
            r       = challenges[round_idx]
            stacked = mle_update_32_v2(evens, odds, r, q)   # (V, N_k//2)

    claim0 = mod_add_32_v2(all_round_evals[0][0], all_round_evals[0][1], q)
    round_evals = jnp.stack([jnp.stack(g_t) for g_t in all_round_evals])
    return claim0, round_evals


c0, re = sumcheck_v4(TINY_TABLES, q=Q_tiny, expression=[['a']],
                     challenges=TINY_CHALS, num_rounds=3)
check(c0, re, "v4")

print("\nBenchmark:")
ms_v4, _ = bench(
    sumcheck_v4, BENCH_TABLES, q=Q32, expression=BENCH_EXPR,
    challenges=BENCH_CHALS, num_rounds=N_BENCH,
    label="v4 reshape split"
)
print(f"  Speedup vs v0: {ms_v0/ms_v4:.2f}×  vs v3: {ms_v3/ms_v4:.2f}×")

---
## Level 5 — `vmap` the Degree Loop

**What changes:** The inner `for t in range(degree+1)` loop is replaced by `jax.vmap`.

**Why it helps:**

Currently we compute `g(0)`, `g(1)`, ..., `g(d)` sequentially — each is a separate kernel launch.
For `degree=3`, that's 4 kernel launches with Python scheduling between them.

With `vmap`, JAX batches all `d+1` evaluations into one kernel.

**Bonus: shared `diff` computation.**  
Inside the MLE update: `diff = (odds - evens) mod q`.  
This `diff` is the same for all `t` values! Without vmap, we recompute it for each `t`.  
With vmap, we compute `diff` once and broadcast over the `t` dimension:

```
Without vmap:  for each t: compute diff (N/2 read) → apply t → sum
               → V × (degree-1) × HBM reads for diff recomputation

With vmap:     compute diff once (N/2 read)
               apply all t values in one batched kernel
               → V × 1 × HBM read for diff (shared across t)
```

In [ ]:
# ── Level 5: vmap the degree loop ─────────────────────────────────────────────

def make_round_evaluator(term_indices, degree):
    """
    Returns a JIT+vmap-compiled function that computes all g(t) for t in [0..degree]
    simultaneously, sharing the diff computation across t values.
    """
    n_eval  = degree + 1
    t_vals  = jnp.arange(n_eval, dtype=jnp.uint32)   # [0, 1, ..., degree]

    def g_at_t_from_diff(t, evens64, diff64, q64):
        """
        Compute g(t) given pre-computed diff = (odds - evens) mod q.
        tables_t = diff * t + evens  mod q
        """
        tables_t = (diff64 * t.astype(jnp.uint64) % q64 + evens64) % q64  # (V, N/2)
        tables_t_u32 = tables_t.astype(jnp.uint32)
        vals = jnp.zeros(tables_t_u32.shape[1], dtype=jnp.uint64)
        for term in term_indices:
            tv = tables_t[term[0]]
            for idx in term[1:]:
                tv = (tv * tables_t[idx]) % q64
            vals = (vals + tv) % q64
        return (jnp.sum(vals) % q64).astype(jnp.uint32)

    @jax.jit
    def compute_all_g(evens, odds, q):
        """
        Compute g(0), g(1), ..., g(degree) in one batched call.
        evens, odds: (V, N//2) uint32
        Returns: (n_eval,) uint32
        """
        q64     = jnp.uint64(q)
        evens64 = evens.astype(jnp.uint64)
        odds64  = odds.astype(jnp.uint64)
        diff64  = (odds64 + q64 - evens64) % q64   # computed ONCE, shared across t

        # vmap maps g_at_t_from_diff over the t_vals dimension
        batched = jax.vmap(
            lambda t: g_at_t_from_diff(t, evens64, diff64, q64)
        )
        return batched(t_vals)   # (n_eval,) uint32

    return compute_all_g


def sumcheck_v5(eval_tables, *, q, expression, challenges, num_rounds):
    """Level 5: vmap over degree loop — g(0)..g(d) computed simultaneously."""
    var_order    = sorted(eval_tables.keys())
    term_indices = build_term_indices(expression, var_order)
    stacked      = stack_tables(eval_tables, var_order)   # (V, N)
    degree       = max(len(term) for term in expression)
    V            = stacked.shape[0]
    q64          = jnp.uint64(q)

    compute_all_g = make_round_evaluator(term_indices, degree)
    all_round_evals = []

    for round_idx in range(num_rounds):
        N_k  = stacked.shape[1]
        view = stacked.reshape(V, N_k // 2, 2)
        evens = view[:, :, 0]
        odds  = view[:, :, 1]

        # All g(t) in one call
        round_evals_row = compute_all_g(evens, odds, q)   # (degree+1,)
        all_round_evals.append(round_evals_row)

        if round_idx < num_rounds - 1:
            r       = challenges[round_idx]
            stacked = mle_update_32_v2(evens, odds, r, q)

    claim0 = mod_add_32_v2(all_round_evals[0][0], all_round_evals[0][1], q)
    round_evals = jnp.stack(all_round_evals)   # (num_rounds, degree+1)
    return claim0, round_evals


# Verify
c0, re = sumcheck_v5(TINY_TABLES, q=Q_tiny, expression=[['a']],
                     challenges=TINY_CHALS, num_rounds=3)
check(c0, re, "v5")

# Show per-round speedup for degree loop
N_vmap = 2**18
stk_vmap = {v: jax.random.randint(key, (N_vmap,), 0, int(Q32), dtype=jnp.uint32)
            for v in ['a','b','c']}
chals_vmap = jax.random.randint(key, (N_BENCH-1,), 1, int(Q32), dtype=jnp.uint32)

print("\nBenchmark:")
ms_v5, _ = bench(
    sumcheck_v5, BENCH_TABLES, q=Q32, expression=BENCH_EXPR,
    challenges=BENCH_CHALS, num_rounds=N_BENCH,
    label="v5 vmap degree loop"
)
print(f"  Speedup vs v0: {ms_v0/ms_v5:.2f}×  vs v4: {ms_v4/ms_v5:.2f}×")

---
## Level 6 — JIT the Full Round Body

**What changes:** The inner computation of each round (split + eval-all-t + fold)
is wrapped in a single `@jax.jit`-compiled function.

**Why this is bigger than it sounds:**

Even with JIT on primitives and vmap, each Python-level operation still dispatches
through the JAX dispatch layer. For one round with `V=3, degree=3`:
- 1 reshape (dispatch)
- 2 slices (dispatch × 2)
- 1 vmap call (dispatch)
- 1 mle_update fold (dispatch)
= ~5 dispatcher round-trips per round × 20 rounds = 100 dispatcher calls

When JIT-compiled as one function:
- XLA sees the ENTIRE round as one computation graph
- `diff` computed inside vmap is reused for the fold (no second read of evens/odds!)
- All operations fuse into as few GPU kernels as possible

```
Before (5 dispatches/round):   diff is computed twice — once in vmap, once in fold
After  (1 dispatch/round):     diff is computed once, reused for both vmap and fold
                               Saves V × N/2 × 4 bytes per round
```

In [ ]:
# ── Level 6: JIT the entire round ─────────────────────────────────────────────

def make_jit_round(term_indices, degree):
    """
    Returns a JIT-compiled function for one full SumCheck round.
    Inputs:  stacked (V, N), challenge r, q
    Outputs: (degree+1,) round_evals, new (V, N//2) stacked tables
    """
    n_eval = degree + 1
    t_vals = jnp.arange(n_eval, dtype=jnp.uint32)

    @jax.jit
    def one_round(stacked, r, q):
        """
        One SumCheck round — single JIT kernel covering:
          1. Reshape split (coalesced)
          2. Compute diff once
          3. vmap: g(t) for all t using shared diff
          4. MLE fold reusing same diff
        """
        V, N_k = stacked.shape
        half   = N_k // 2
        q64    = jnp.uint64(q)

        # Coalesced split
        view   = stacked.reshape(V, half, 2)
        evens  = view[:, :, 0].astype(jnp.uint64)   # (V, half)
        odds   = view[:, :, 1].astype(jnp.uint64)   # (V, half)

        # diff computed ONCE — shared by vmap g(t) evals AND the fold below
        diff = (odds + q64 - evens) % q64            # (V, half)

        # Evaluate expression g(t) for all t using shared diff
        def g_at_t(t):
            tables_t = (diff * t.astype(jnp.uint64) % q64 + evens) % q64
            vals = jnp.zeros(half, dtype=jnp.uint64)
            for term in term_indices:
                tv = tables_t[term[0]]
                for idx in term[1:]:
                    tv = (tv * tables_t[idx]) % q64
                vals = (vals + tv) % q64
            return (jnp.sum(vals) % q64).astype(jnp.uint32)

        round_evals = jax.vmap(g_at_t)(t_vals)   # (n_eval,)

        # Fold: reuse diff — no re-read of evens/odds from HBM
        r64       = r.astype(jnp.uint64)
        new_stacked = (diff * r64 % q64 + evens) % q64   # (V, half)
        new_stacked = new_stacked.astype(jnp.uint32)

        return round_evals, new_stacked

    return one_round


def sumcheck_v6(eval_tables, *, q, expression, challenges, num_rounds):
    """Level 6: JIT the full round — single kernel covers split+vmap+fold."""
    var_order    = sorted(eval_tables.keys())
    term_indices = build_term_indices(expression, var_order)
    stacked      = stack_tables(eval_tables, var_order).astype(jnp.uint32)
    degree       = max(len(term) for term in expression)

    one_round = make_jit_round(term_indices, degree)
    all_round_evals = []

    for round_idx in range(num_rounds):
        r = challenges[round_idx] if round_idx < num_rounds - 1 else jnp.uint32(0)
        round_row, stacked = one_round(stacked, r, q)
        all_round_evals.append(round_row)

    claim0 = mod_add_32_v2(all_round_evals[0][0], all_round_evals[0][1], q)
    round_evals = jnp.stack(all_round_evals)
    return claim0, round_evals


c0, re = sumcheck_v6(TINY_TABLES, q=Q_tiny, expression=[['a']],
                     challenges=TINY_CHALS, num_rounds=3)
check(c0, re, "v6")

print("\nBenchmark:")
ms_v6, _ = bench(
    sumcheck_v6, BENCH_TABLES, q=Q32, expression=BENCH_EXPR,
    challenges=BENCH_CHALS, num_rounds=N_BENCH,
    label="v6 jit full round"
)
print(f"  Speedup vs v0: {ms_v0/ms_v6:.2f}×  vs v5: {ms_v5/ms_v6:.2f}×")

---
## Level 7 — JIT the Entire `sumcheck_32` Function

**What changes:** The entire `sumcheck_32` including the outer Python for loop
is wrapped in `@jax.jit`.

**How it works:**  
When JAX JITs a function containing a Python `for` loop, it **unrolls** the loop at trace time.
Each iteration becomes a block of XLA operations. The result is one giant XLA program
covering all `n` rounds.

**Pros:**
- Zero Python overhead at runtime (all n rounds execute inside one XLA call)
- XLA can optimize across rounds — e.g., pipeline memory accesses between consecutive rounds
- No per-round Python dispatch cost

**Cons:**
- Compile time is `O(n)` — for `n=20`, trace time grows linearly
- The compiled XLA program is large; for `n≥24` it can be slow to compile
- Use Level 8 (`lax.scan`) for very large `n` where compile time matters

```
Level 6 Python for loop:  20 JIT calls × ~2µs dispatch = ~40µs Python overhead
Level 7 one JIT call:     1 JIT call   × ~2µs dispatch = ~2µs Python overhead
                          XLA runs all 20 rounds as one fused program
```

In [ ]:
# ── Level 7: JIT the entire sumcheck function ──────────────────────────────────

def make_sumcheck_jit(expression, num_rounds):
    """
    Factory: bake expression and num_rounds as static values at trace time.
    Returns a JIT-compiled sumcheck function.
    
    Using a factory instead of static_argnames lets XLA specialize the
    kernel for this specific expression structure — tighter loop unrolling.
    """
    var_order    = sorted(set(v for term in expression for v in term))
    term_indices = build_term_indices(expression, var_order)
    degree       = max(len(term) for term in expression)
    n_eval       = degree + 1
    t_vals       = jnp.arange(n_eval, dtype=jnp.uint32)

    @jax.jit
    def sumcheck(stacked, q, challenges):
        """
        stacked:    (V, 2^num_rounds) uint32 — tables in var_order row order
        q:          uint32 scalar
        challenges: (num_rounds-1,) uint32
        """
        q64 = jnp.uint64(q)
        V   = stacked.shape[0]
        all_round_evals = []

        # Python loop — unrolled at trace time into one XLA program
        for round_idx in range(num_rounds):
            N_k  = stacked.shape[1]
            half = N_k // 2

            # Coalesced split
            view  = stacked.reshape(V, half, 2)
            evens = view[:, :, 0].astype(jnp.uint64)
            odds  = view[:, :, 1].astype(jnp.uint64)
            diff  = (odds + q64 - evens) % q64

            def g_at_t(t):
                tables_t = (diff * t.astype(jnp.uint64) % q64 + evens) % q64
                vals = jnp.zeros(half, dtype=jnp.uint64)
                for term in term_indices:
                    tv = tables_t[term[0]]
                    for idx in term[1:]:
                        tv = (tv * tables_t[idx]) % q64
                    vals = (vals + tv) % q64
                return (jnp.sum(vals) % q64).astype(jnp.uint32)

            round_row = jax.vmap(g_at_t)(t_vals)
            all_round_evals.append(round_row)

            # Fold
            if round_idx < num_rounds - 1:
                r64     = challenges[round_idx].astype(jnp.uint64)
                stacked = ((diff * r64 % q64 + evens) % q64).astype(jnp.uint32)

        round_evals = jnp.stack(all_round_evals)   # (num_rounds, n_eval)
        claim0 = ((round_evals[0, 0].astype(jnp.uint64) +
                   round_evals[0, 1].astype(jnp.uint64)) % q64).astype(jnp.uint32)
        return claim0, round_evals

    return sumcheck, var_order


def sumcheck_v7(eval_tables, *, q, expression, challenges, num_rounds):
    """Level 7: entire sumcheck JIT-compiled — n rounds as one XLA program."""
    sc_fn, var_order = make_sumcheck_jit(expression, num_rounds)
    stacked = stack_tables(eval_tables, var_order).astype(jnp.uint32)
    return sc_fn(stacked, q, challenges)


# Note: make_sumcheck_jit caches compiled versions per (expression, num_rounds) pair.
# Production usage: call make_sumcheck_jit once, reuse the returned function.

c0, re = sumcheck_v7(TINY_TABLES, q=Q_tiny, expression=[['a']],
                     challenges=TINY_CHALS, num_rounds=3)
check(c0, re, "v7")

# Pre-compile (first call triggers JIT trace)
sc_fn_bench, var_order_bench = make_sumcheck_jit(BENCH_EXPR, N_BENCH)
stacked_bench = stack_tables(BENCH_TABLES, var_order_bench).astype(jnp.uint32)

print("\nBenchmark (note: first call compiles XLA program for all 20 rounds):")
ms_v7, _ = bench(
    sc_fn_bench, stacked_bench, Q32, BENCH_CHALS,
    label="v7 full JIT sumcheck"
)
print(f"  Speedup vs v0: {ms_v0/ms_v7:.2f}×  vs v6: {ms_v6/ms_v7:.2f}×")

---
## Level 8 — `lax.scan` for the Outer Round Loop

**What changes:** Python `for round_idx in range(num_rounds)` is replaced by `lax.scan`.

**Why this is different from Level 7:**  
Level 7 unrolls all `n` rounds at trace time → compile time is `O(n)`.  
Level 8 compiles only ONE round body and repeats it `n` times at runtime.

- Level 7 is better when `n` is small (faster compile, XLA can optimize across rounds)  
- Level 8 is better when `n` is large (constant compile time regardless of `n`)

**The shape problem and how we solve it:**  
`lax.scan` requires the carry to have a **fixed shape** at every step.  
Our tables shrink by half each round: `(V, N) → (V, N/2) → (V, N/4) → ...`  
This violates the fixed-shape requirement!

**Solution: fixed-size carry with active-length tracking**

```
Carry = (buffer of shape (V, N), active_len scalar)

Round k:
  - Extract active slice: buffer[:, :active_len]
  - Fold: compute new (V, active_len//2) table
  - Write result back into buffer[:, :active_len//2]
  - Return (buffer, active_len//2) as new carry
```

The buffer stays at size `(V, N)` — only the first `active_len` elements are valid.

In [ ]:
# ── Level 8: lax.scan outer loop with fixed-size carry ────────────────────────

def make_sumcheck_scan(expression, num_rounds):
    """
    Factory: returns a JIT+scan compiled sumcheck.
    Compile time is O(1) in num_rounds — only the round body is compiled once.
    """
    var_order    = sorted(set(v for term in expression for v in term))
    term_indices = build_term_indices(expression, var_order)
    degree       = max(len(term) for term in expression)
    n_eval       = degree + 1
    t_vals       = jnp.arange(n_eval, dtype=jnp.uint32)

    @jax.jit
    def sumcheck_scan(stacked, q, challenges):
        """
        stacked:    (V, N) uint32 where N = 2^num_rounds
        q:          uint32 scalar
        challenges: (num_rounds-1,) uint32 — padded to (num_rounds,) internally
        """
        q64 = jnp.uint64(q)
        V, N = stacked.shape

        # Pad challenges to length num_rounds (last element unused — fold skipped)
        r_padded = jnp.concatenate(
            [challenges, jnp.zeros(num_rounds - challenges.shape[0], dtype=jnp.uint32)]
        )

        def round_body(carry, r):
            """
            carry = (buffer (V, N), active_len scalar int32)
            r     = challenge for this round (uint32)
            Returns: (new_carry, round_evals_row)
            """
            buffer, active_len = carry
            half = active_len // 2

            # Extract active slice using dynamic slice (shape-preserving)
            active = lax.dynamic_slice_in_dim(buffer, 0, active_len, axis=1)  # (V, active_len)

            # Reshape split on the active slice
            active_view = active.reshape(V, half, 2)
            evens = active_view[:, :, 0].astype(jnp.uint64)   # (V, half)
            odds  = active_view[:, :, 1].astype(jnp.uint64)   # (V, half)
            diff  = (odds + q64 - evens) % q64                # computed once

            # vmap: compute all g(t) using shared diff
            def g_at_t(t):
                tables_t = (diff * t.astype(jnp.uint64) % q64 + evens) % q64
                vals = jnp.zeros(half, dtype=jnp.uint64)
                for term in term_indices:
                    tv = tables_t[term[0]]
                    for idx in term[1:]:
                        tv = (tv * tables_t[idx]) % q64
                    vals = (vals + tv) % q64
                return (jnp.sum(vals) % q64).astype(jnp.uint32)

            round_evals_row = jax.vmap(g_at_t)(t_vals)   # (n_eval,)

            # Fold: reuse diff
            r64     = r.astype(jnp.uint64)
            new_active = ((diff * r64 % q64 + evens) % q64).astype(jnp.uint32)  # (V, half)

            # Write folded result back into first half of buffer (in-place update)
            new_buffer = lax.dynamic_update_slice_in_dim(
                buffer, new_active, 0, axis=1
            )  # (V, N) — only first half updated

            new_carry = (new_buffer, half)
            return new_carry, round_evals_row

        init_carry = (stacked, N)
        (_, _), all_round_evals = lax.scan(round_body, init_carry, r_padded)
        # all_round_evals: (num_rounds, n_eval)

        claim0 = ((all_round_evals[0, 0].astype(jnp.uint64) +
                   all_round_evals[0, 1].astype(jnp.uint64)) % q64).astype(jnp.uint32)
        return claim0, all_round_evals

    return sumcheck_scan, var_order


def sumcheck_v8(eval_tables, *, q, expression, challenges, num_rounds):
    """Level 8: lax.scan outer loop — compile time O(1) in num_rounds."""
    sc_fn, var_order = make_sumcheck_scan(expression, num_rounds)
    stacked = stack_tables(eval_tables, var_order).astype(jnp.uint32)
    return sc_fn(stacked, q, challenges)


c0, re = sumcheck_v8(TINY_TABLES, q=Q_tiny, expression=[['a']],
                     challenges=TINY_CHALS, num_rounds=3)
check(c0, re, "v8")

# Pre-compile once
sc_fn_scan, var_order_scan = make_sumcheck_scan(BENCH_EXPR, N_BENCH)
stacked_bench_scan = stack_tables(BENCH_TABLES, var_order_scan).astype(jnp.uint32)

print("\nBenchmark:")
ms_v8, _ = bench(
    sc_fn_scan, stacked_bench_scan, Q32, BENCH_CHALS,
    label="v8 lax.scan outer loop"
)
print(f"  Speedup vs v0: {ms_v0/ms_v8:.2f}×  vs v7: {ms_v7/ms_v8:.2f}×")
print()
print("  Trade-off: v7 (unrolled) vs v8 (scan):")
print(f"    v7 ms: {ms_v7:.2f}  v8 ms: {ms_v8:.2f}")
print("    v7 wins at small n (XLA can optimize across rounds)")
print("    v8 wins at large n (compile time stays constant)")

---
## Level 9 — Production: All Optimizations Combined

This is the final implementation that ships. It:

1. Uses the stacked `(V, N)` SoA layout (Level 3)
2. Uses reshape split for coalesced access (Level 4)
3. Computes `diff` once, reuses it for vmap **and** fold (Level 6)
4. Uses `vmap` over degree evaluation points (Level 5)
5. JITs the entire function — zero Python overhead at runtime (Level 7)
6. Selects between JIT-unroll vs `lax.scan` based on `num_rounds`:
   - `num_rounds ≤ 20`: use JIT unroll (Level 7) — faster runtime
   - `num_rounds > 20`: use `lax.scan` (Level 8) — faster compile

Additionally:
- All arithmetic stays in `uint64` until the final output cast → no extra mod operations
- `challenges` padding is done once outside the hot loop
- `term_indices` and `var_order` are baked in as compile-time constants

In [ ]:
# ── Level 9: Production implementation ────────────────────────────────────────

_COMPILED_CACHE = {}

def _compile_sumcheck(expression, num_rounds, use_scan=None):
    """
    Compile and cache a sumcheck function for the given (expression, num_rounds).
    Automatically chooses JIT-unroll (n≤20) or lax.scan (n>20) unless overridden.
    """
    cache_key = (tuple(tuple(t) for t in expression), num_rounds)
    if cache_key in _COMPILED_CACHE:
        return _COMPILED_CACHE[cache_key]

    if use_scan is None:
        use_scan = (num_rounds > 20)

    if use_scan:
        fn, var_order = make_sumcheck_scan(expression, num_rounds)
    else:
        fn, var_order = make_sumcheck_jit(expression, num_rounds)

    _COMPILED_CACHE[cache_key] = (fn, var_order)
    return fn, var_order


def sumcheck_v9(eval_tables, *, q, expression, challenges, num_rounds):
    """
    Level 9 — Production SumCheck prover.

    Drop-in replacement for the baseline sumcheck_32.
    Same API, all optimizations applied automatically.

    Parameters
    ----------
    eval_tables : dict[str, jnp.ndarray] of uint32 — one table per variable
    q           : uint32 scalar — prime modulus
    expression  : list[list[str]] — polynomial expression
    challenges  : (num_rounds-1,) uint32 array — verifier challenges
    num_rounds  : int — number of SumCheck rounds

    Returns
    -------
    (claim0, round_evals)
      claim0      : uint32 scalar
      round_evals : (num_rounds, degree+1) uint32 array
    """
    fn, var_order = _compile_sumcheck(expression, num_rounds)
    stacked = stack_tables(eval_tables, var_order).astype(jnp.uint32)
    return fn(stacked, q, challenges)


# ── Full correctness sweep: all 4 expressions ─────────────────────────────────
print("Correctness sweep (2-variable tables, Q=17):")
Q_check  = jnp.uint32(17)
tbls_chk = {
    'a': jnp.array([1, 2, 3, 4], dtype=jnp.uint32),
    'b': jnp.array([2, 3, 1, 2], dtype=jnp.uint32),
    'c': jnp.array([0, 1, 2, 3], dtype=jnp.uint32),
}
chals_chk = jnp.array([7], dtype=jnp.uint32)

for expr_str, expr in [('a',         [['a']]),
                        ('a*b',       [['a','b']]),
                        ('a*b + c',   [['a','b'],['c']]),
                        ('a*b*c',     [['a','b','c']])]:
    tbls_used = {v: tbls_chk[v] for v in set(v for t in expr for v in t)}

    c0_ref, re_ref = sumcheck_v0(tbls_used, q=Q_check, expression=expr,
                                  challenges=chals_chk, num_rounds=2)
    c0_v9,  re_v9  = sumcheck_v9(tbls_used, q=Q_check, expression=expr,
                                  challenges=chals_chk, num_rounds=2)

    match = (int(c0_ref)==int(c0_v9)) and (re_ref.tolist()==re_v9.tolist())
    print(f"  {expr_str:10s}  claim0={int(c0_v9)}  {'PASS' if match else 'FAIL'}")


# ── Benchmark v9 vs baseline ──────────────────────────────────────────────────
print("\nFinal benchmark (n=20, a*b*c):")
ms_v9, _ = bench(
    sumcheck_v9, BENCH_TABLES, q=Q32, expression=BENCH_EXPR,
    challenges=BENCH_CHALS, num_rounds=N_BENCH,
    label="v9 production"
)
print(f"  Total speedup vs baseline: {ms_v0/ms_v9:.1f}×")

---
## Full Benchmark Suite — All Levels, All Expressions

In [ ]:
# ── Benchmark suite: all 9 levels × all 4 expressions ─────────────────────────

EXPRESSIONS = [
    ('a',       [['a']]),
    ('a*b',     [['a', 'b']]),
    ('a*b+c',   [['a', 'b'], ['c']]),
    ('a*b*c',   [['a', 'b', 'c']]),
]

VERSIONS = [
    ('v0 baseline',  sumcheck_v0),
    ('v1 jit prim',  sumcheck_v1),
    ('v2 fused mle', sumcheck_v2),
    ('v3 stacked',   sumcheck_v3),
    ('v4 reshape',   sumcheck_v4),
    ('v5 vmap',      sumcheck_v5),
    ('v6 jit round', sumcheck_v6),
    ('v7 jit full',  sumcheck_v7),
    ('v8 scan',      sumcheck_v8),
    ('v9 production',sumcheck_v9),
]

N_SUITE = 18   # n=18 keeps the benchmark fast; scale up for production
N_suite = 2**N_SUITE

tables_suite = {v: jax.random.randint(key, (N_suite,), 0, int(Q32), dtype=jnp.uint32)
                for v in ['a', 'b', 'c']}
chals_suite  = jax.random.randint(key, (N_SUITE-1,), 1, int(Q32), dtype=jnp.uint32)

print(f"Benchmark suite: n={N_SUITE}, N=2^{N_SUITE}={N_suite:,}")
print(f"{'Version':20s}", end="")
for name, _ in EXPRESSIONS:
    print(f"  {name:10s}", end="")
print("  Speedup vs v0")
print("-" * 80)

baseline_ms = {}
results_table = {}

RUNS_SUITE = 5
WARMUP_SUITE = 2

for ver_name, ver_fn in VERSIONS:
    row_times = []
    for expr_name, expr in EXPRESSIONS:
        tbls = {v: tables_suite[v] for v in set(v for t in expr for v in t)}
        try:
            for _ in range(WARMUP_SUITE):
                c0_w, re_w = ver_fn(tbls, q=Q32, expression=expr,
                                    challenges=chals_suite, num_rounds=N_SUITE)
                re_w.block_until_ready()
            t0 = time.perf_counter()
            for _ in range(RUNS_SUITE):
                c0_w, re_w = ver_fn(tbls, q=Q32, expression=expr,
                                    challenges=chals_suite, num_rounds=N_SUITE)
                re_w.block_until_ready()
            ms = (time.perf_counter()-t0)/RUNS_SUITE*1000
        except Exception as e:
            ms = float('nan')
        row_times.append(ms)
        if ver_name == 'v0 baseline':
            baseline_ms[expr_name] = ms

    results_table[ver_name] = row_times
    avg_speedup = sum(
        baseline_ms.get(en, 1) / rt for (en, _), rt in zip(EXPRESSIONS, row_times)
        if rt == rt  # not nan
    ) / len(EXPRESSIONS)

    print(f"{ver_name:20s}", end="")
    for ms in row_times:
        if ms == ms:
            print(f"  {ms:8.1f}ms", end="")
        else:
            print(f"  {'ERR':>10s}", end="")
    print(f"  {avg_speedup:5.1f}×")

---
## Optimization Ladder: Cumulative Speedup Breakdown

In [ ]:
# ── Cumulative speedup chart ───────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# Use a*b*c results (most interesting — degree=3, 3 variables)
abc_idx = [n for n, _ in EXPRESSIONS].index('a*b*c')
ver_names_short = [v.split(' ')[0] + '\n' + ' '.join(v.split(' ')[1:])
                   for v, _ in VERSIONS]
abc_times = [results_table[v][abc_idx] for v, _ in VERSIONS]
baseline_t = abc_times[0]
speedups = [baseline_t / t if t == t else 0 for t in abc_times]

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(VERSIONS)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Latency bar
bars = ax1.bar(range(len(VERSIONS)), abc_times, color=colors, edgecolor='black', lw=0.5)
ax1.set_xticks(range(len(VERSIONS)))
ax1.set_xticklabels([v.split(' ')[0] for v, _ in VERSIONS], rotation=45, ha='right', fontsize=9)
ax1.set_ylabel('Latency (ms)')
ax1.set_title(f'SumCheck Latency by Level (n={N_SUITE}, a*b*c)')
ax1.set_yscale('log')
for bar, ms in zip(bars, abc_times):
    if ms == ms:
        ax1.text(bar.get_x()+bar.get_width()/2, ms*1.05, f'{ms:.0f}ms',
                 ha='center', va='bottom', fontsize=7, rotation=90)

# Speedup bar
sbars = ax2.bar(range(len(VERSIONS)), speedups, color=colors, edgecolor='black', lw=0.5)
ax2.set_xticks(range(len(VERSIONS)))
ax2.set_xticklabels([v.split(' ')[0] for v, _ in VERSIONS], rotation=45, ha='right', fontsize=9)
ax2.axhline(1, color='red', linestyle='--', alpha=0.5, label='Baseline')
ax2.set_ylabel('Speedup vs Baseline')
ax2.set_title(f'Cumulative Speedup (n={N_SUITE}, a*b*c)')
for bar, sp in zip(sbars, speedups):
    if sp > 0:
        ax2.text(bar.get_x()+bar.get_width()/2, sp+0.02, f'{sp:.1f}×',
                 ha='center', va='bottom', fontsize=8)
ax2.legend()

plt.tight_layout()
plt.show()

print("\nOptimization ladder (a*b*c expression):")
print(f"{'Level':20s}  {'Latency':>10s}  {'Speedup':>10s}  {'Delta':>8s}  What changed")
print("-"*80)
prev_sp = 1.0
changes = [
    "Reference",
    "@jax.jit on mod_add/sub/mul",
    "Fuse sub+mul+add in one JIT expr",
    "Dict → (V,N) stacked array",
    "[0::2] → reshape(-1,2)[:,0]",
    "for t loop → jax.vmap, shared diff",
    "JIT split+vmap+fold as one kernel",
    "JIT entire sumcheck (loop unroll)",
    "lax.scan outer loop (fixed buffer)",
    "Auto-select v7/v8 + compile cache",
]
for (ver_name, _), ms, sp, chg in zip(VERSIONS, abc_times, speedups, changes):
    delta = f"{sp/prev_sp:.2f}×" if sp > 0 else "  ERR"
    print(f"{ver_name:20s}  {ms:8.1f} ms  {sp:8.1f}×  {delta:>8s}  {chg}")
    prev_sp = sp if sp > 0 else prev_sp

---
## HBM Traffic Analysis — Where the Bytes Go

In [ ]:
# ── HBM traffic model: how each optimization reduces bytes moved ───────────────

def model_hbm_bytes(n, V, degree, variant):
    """
    Estimate total HBM bytes moved for one sumcheck run.
    variant: 'baseline', 'fused_mle', 'reshape', 'vmap_shared_diff', 'full_fused'
    """
    B = 4   # uint32 = 4 bytes
    total = 0

    for k in range(n):
        Nk   = (2**n) >> k
        half = Nk >> 1

        if variant == 'baseline':
            total += 2 * V * Nk * B        # stride-2 gather: read full table, write 2 halves
            total += 9 * V * half * B * max(0, degree-1)  # unfused mle_update for t≥2
            total += V * half * B * (degree+1) * 2        # eval: read + write per t
            total += half * B * (degree+1)                 # reduce per t
            total += 9 * V * half * B                      # fold (unfused)

        elif variant == 'fused_mle':
            total += 2 * V * Nk * B                       # stride-2 gather still
            total += 3 * V * half * B * max(0, degree-1)  # fused mle_update: 3 passes
            total += V * half * B * (degree+1) * 2        # eval read+write per t
            total += half * B * (degree+1)                 # reduce per t
            total += 3 * V * half * B                      # fold fused

        elif variant == 'reshape':
            total += V * Nk * B                            # reshape view: 1 read of full table
            total += 3 * V * half * B * max(0, degree-1)  # fused mle_update
            total += V * half * B * (degree+1) * 2        # eval
            total += half * B * (degree+1)                 # reduce
            total += 3 * V * half * B                      # fold

        elif variant == 'vmap_shared_diff':
            total += V * Nk * B                            # reshape view
            total += 2 * V * half * B                      # read evens+odds for diff (once)
            total += V * half * B * (degree+1)             # tables_t per t (from diff in regs)
            total += half * B * (degree+1)                 # reduce per t
            total += V * half * B                          # write fold output

        elif variant == 'full_fused':
            # XLA fuses everything: diff stays in registers for vmap AND fold
            total += V * Nk * B                            # one read of full table
            total += half * B * (degree+1)                 # reduce per t (reg → scalar)
            total += V * half * B                          # write fold output

    return total


n, V, degree = 20, 3, 3
variants = [
    ('baseline (v0)',           'baseline'),
    ('+ fused MLE (v2)',        'fused_mle'),
    ('+ reshape split (v4)',    'reshape'),
    ('+ shared diff / vmap (v5)', 'vmap_shared_diff'),
    ('+ full JIT fusion (v7)',  'full_fused'),
]

base_bytes = model_hbm_bytes(n, V, degree, 'baseline')

print(f"HBM Traffic Model: n={n}, V={V}, degree={degree} (a*b*c)")
print(f"{'Variant':40s}  {'Total GB':>10s}  {'Reduction':>10s}  {'vs prev':>8s}")
print("-"*72)

prev_bytes = base_bytes
for label, variant in variants:
    bytes_ = model_hbm_bytes(n, V, degree, variant)
    red    = base_bytes / bytes_
    delta  = prev_bytes / bytes_
    print(f"  {label:38s}  {bytes_/1e9:8.2f} GB  {red:8.2f}×    {delta:6.2f}×")
    prev_bytes = bytes_

print(f"\n  Baseline peak bandwidth used   ≈ {base_bytes/1e9:.1f} GB")
final_bytes = model_hbm_bytes(n, V, degree, 'full_fused')
print(f"  Full-fused peak bandwidth used ≈ {final_bytes/1e9:.1f} GB")
print(f"  Net HBM traffic reduction: {base_bytes/final_bytes:.1f}×")

---
## Summary

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  OPTIMIZATION SUMMARY                                                       │
├────────────┬──────────────────────────┬───────────────┬─────────────────────┤
│  Level     │  Technique               │  HBM Traffic  │  Why It Helps       │
├────────────┼──────────────────────────┼───────────────┼─────────────────────┤
│  v0 Base   │  student.py as-is        │  Reference    │  —                  │
│  v1 JIT    │  @jax.jit primitives     │  Same         │  Less Python        │
│            │                          │               │  dispatch overhead  │
│  v2 Fused  │  MLE: one JIT expression │  ÷3           │  3 kernels → 1      │
│  MLE       │  for sub+mul+add         │               │  biggest BW win     │
│  v3 Stack  │  dict → (V,N) array      │  Same         │  Enables clean JIT  │
│            │                          │               │  tracing            │
│  v4 Reshape│  [0::2] → reshape[:,0]   │  ÷1.5         │  Coalesced GPU      │
│            │                          │               │  memory access      │
│  v5 vmap   │  jax.vmap degree loop    │  ÷1.5–2       │  Shared diff,       │
│            │                          │               │  batch kernel       │
│  v6 JIT    │  JIT split+vmap+fold     │  ÷1.3         │  diff reused for    │
│  Round     │  as one kernel           │               │  vmap AND fold      │
│  v7 JIT    │  JIT entire function     │  Same         │  Zero Python        │
│  Full      │  (loop unroll at trace)  │               │  overhead at run    │
│  v8 Scan   │  lax.scan outer loop     │  Same         │  O(1) compile time  │
│            │  (fixed-buffer carry)    │               │  for large n        │
│  v9 Prod   │  All + auto select +     │  All gains    │  Best of v7 & v8    │
│            │  compile cache           │  stacked      │  depending on n     │
├────────────┴──────────────────────────┴───────────────┴─────────────────────┤
│  FUNDAMENTAL LIMITS (cannot be optimized away):                             │
│  • Sequential rounds: each needs previous round's folded table              │
│  • Memory-bound: AI ≈ 0.17 FLOPs/byte — far below GPU ridge point          │
│  • Modular arithmetic: no native int mod on GPU — uint64 intermediate needed│
└─────────────────────────────────────────────────────────────────────────────┘
```

### Key takeaway

SumCheck is memory-bandwidth bound. The optimization path is:

1. **Reduce bytes moved** (fusion, layout, reshape) — biggest wins
2. **Maximize effective bandwidth** (coalescing, cache-friendly access)
3. **Eliminate software overhead** (jax.jit, vmap, lax.scan)
4. **Scale out** (multi-GPU via `pmap`) when single-device bandwidth saturates

The `n`-round outer loop is **unavoidably sequential** — focus GPU effort on the
table operations inside each round, not on parallelizing across rounds.